<a href="https://colab.research.google.com/github/SeoYun-Y/thinfiler-credit-scoring/blob/track-b%2F03-alt-features/notebooks/track-b/track_b_02_%E1%84%83%E1%85%A2%E1%84%8B%E1%85%A1%E1%86%AB%E1%84%87%E1%85%A7%E1%86%AB%E1%84%89%E1%85%AE_%E1%84%89%E1%85%A5%E1%86%AB%E1%84%8C%E1%85%A5%E1%86%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Track B(Sheet11, 통신카드CB결합정보) 대안변수 선정 과정

담당: 서윤 | 선행 노트북: track_b_01_타겟변수_검증.ipynb

본 노트북은 코드북 738개 필드를 전수 재스캔하고, 정보가치(IV) 기준 필터링·다중공선성 제거·순환논리 검증·씬파일러 세그먼트 유효성 점검을 거쳐 최종 대안변수(X)를 확정하는 전체 과정을 담는다. 마지막으로 확정된 변수로 로지스틱회귀 스모크 테스트까지 수행한다.

## 1. 코드북 전체 필드 재스캔

Sheet11 738개 필드를 불러오고, 씬파일러 정의 필드(5개)·식별자·타겟/연체 관련 필드(8개)를 사전 제외한다. 순환논리 방지와 데이터 누출(leakage) 방지 목적이다. 이후 202103 기준으로 결측률·최빈값비율·고유값수를 배치 스캔한다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

base_path = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터'
scan_path = f'{base_path}/통신카드CB_씬파일러.csv'
codebook_path = f'{base_path}/093-117_금융 합성데이터_데이터구성상세.xlsx'

Mounted at /content/drive


In [2]:
sheet11 = pd.read_excel(codebook_path, sheet_name='11.통신카드CB 결합정보')
print(sheet11.shape)
sheet11.head()

(738, 5)


,No,필드명,설명,코드여부,타입
0,1,BASE_YM,기준년월,N,Char
1,2,CUST_ID,고객식별자,N,Char
2,3,SEX,성별,Y,Char
3,4,AGE,연령대,Y,Char
4,5,JB_TP,직업군,Y,Char


In [3]:
exclude_fields = [
    'BASE_YM', 'CUST_ID',
    # 씬파일러 정의 필드
    'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004', 'PYE_C18233005', 'PYE_L10210000',
    # 타겟 및 연체 관련 필드 (leakage 방지)
    'PYE_D1011000C', 'PYE_D10110001', 'PYE_D10110003', 'PYE_D10110006',
    'PYE_D10210D00', 'PYE_D10232000', 'PYE_MAX_DLQ_DAY', 'PYE_SC0000000',
]

all_fields = sheet11['필드명'].tolist()
candidate_fields = [f for f in all_fields if f not in exclude_fields]
print(f"전체 필드: {len(all_fields)}개")
print(f"제외 후 후보: {len(candidate_fields)}개")

전체 필드: 738개
제외 후 후보: 723개


In [4]:
# 중복 필드명 확인
dup_fields = sheet11['필드명'][sheet11['필드명'].duplicated(keep=False)]
print("중복된 필드명:")
print(dup_fields.sort_values())

# 전체 고유값 개수도 확인
print(f"\n전체 행: {len(all_fields)}개")
print(f"고유 필드명: {len(set(all_fields))}개")

중복된 필드명:
Series([], Name: 필드명, dtype: object)

전체 행: 738개
고유 필드명: 738개


In [5]:
import gc

df_path = f'{scan_path}/202103_통신카드CB결합.csv'
batch_size = 20

results = []

for i in range(0, len(candidate_fields), batch_size):
    batch_cols = ['CUST_ID'] + candidate_fields[i:i+batch_size]
    df_batch = pd.read_csv(df_path, usecols=batch_cols)

    for col in candidate_fields[i:i+batch_size]:
        missing_pct = df_batch[col].isnull().mean() * 100
        n_unique = df_batch[col].nunique(dropna=True)
        if n_unique > 0:
            top_ratio = df_batch[col].value_counts(normalize=True, dropna=True).iloc[0] * 100
        else:
            top_ratio = 100.0

        if missing_pct >= 90:
            verdict = '제외(결측90%+)'
        elif n_unique <= 1:
            verdict = '제외(상수)'
        elif top_ratio >= 99:
            verdict = '제외(최빈값99%+)'
        else:
            verdict = '사용가능'

        results.append({
            '변수명': col, '결측률(%)': round(missing_pct, 2),
            '고유값수': n_unique, '최빈값비율(%)': round(top_ratio, 2),
            '판정': verdict
        })

    del df_batch
    gc.collect()
    print(f"[{i+batch_size}/{len(candidate_fields)}] 배치 완료")

result_df = pd.DataFrame(results)
print(f"\n전체 판정 결과:")
print(result_df['판정'].value_counts())

[20/723] 배치 완료
[40/723] 배치 완료
[60/723] 배치 완료
[80/723] 배치 완료
[100/723] 배치 완료
[120/723] 배치 완료
[140/723] 배치 완료
[160/723] 배치 완료
[180/723] 배치 완료
[200/723] 배치 완료
[220/723] 배치 완료
[240/723] 배치 완료
[260/723] 배치 완료
[280/723] 배치 완료
[300/723] 배치 완료
[320/723] 배치 완료
[340/723] 배치 완료
[360/723] 배치 완료
[380/723] 배치 완료
[400/723] 배치 완료
[420/723] 배치 완료
[440/723] 배치 완료
[460/723] 배치 완료
[480/723] 배치 완료
[500/723] 배치 완료
[520/723] 배치 완료
[540/723] 배치 완료
[560/723] 배치 완료
[580/723] 배치 완료
[600/723] 배치 완료
[620/723] 배치 완료
[640/723] 배치 완료
[660/723] 배치 완료
[680/723] 배치 완료
[700/723] 배치 완료
[720/723] 배치 완료
[740/723] 배치 완료

전체 판정 결과:
판정
사용가능           358
제외(결측90%+)     235
제외(최빈값99%+)     78
제외(상수)          52
Name: count, dtype: int64


In [6]:
usable = result_df[result_df['판정']=='사용가능']['변수명'].tolist()
print(f"1차 사용가능 변수: {len(usable)}개")
result_df[result_df['판정']=='사용가능'].sort_values('결측률(%)')

1차 사용가능 변수: 358개


,변수명,결측률(%),고유값수,최빈값비율(%),판정
290,R12M_CLOTHES_AMT,0.00,541,92.80,사용가능
302,R12M_OIL_AMT,0.00,2859,77.04,사용가능
299,R12M_FURN_AMT,0.00,333,98.95,사용가능
298,R12M_MED_AMT,0.00,1724,65.62,사용가능
297,R12M_EDU_AMT,0.00,1072,98.00,사용가능
...,...,...,...,...,...
436,CPYT_TRANS_CPPY_AMT_RT,87.61,2741,24.18,사용가능
637,YOY_YR_SALES_RTC,88.29,38070,1.23,사용가능
567,QOQ_R3M_M_CONV_AMT_RTC,88.69,2186,26.50,사용가능
62,MX_SHP_ADM,88.70,16,26.46,사용가능


**결과**: 723개 후보 중 358개가 사용가능으로 판정되었다.

## 2. 원본변수와 파생변수 구분

필드명 접두사(QOQ_, YOY_, CPYT_)를 기준으로 원본과 파생(변화율)을 구분한다. 파생변수는 데이터셋 제공기관이 미리 계산하여 제공한 필드이다.

In [7]:
def classify_type(col):
    if col.startswith(('QOQ_', 'YOY_', 'CPYT_')):
        return '파생(변화율)'
    else:
        return '원본'

result_df['변수유형'] = result_df['변수명'].apply(classify_type)

usable_df = result_df[result_df['판정']=='사용가능'].copy()
print(usable_df['변수유형'].value_counts())

변수유형
원본         197
파생(변화율)    161
Name: count, dtype: int64


## 3. 파생변수의 결측 패턴 확인 및 처리

파생변수 161개의 결측률이 중앙값 46%로 매우 높게 나타나, 원인을 확인한다. QOQ/YOY/CPYT 세 계열에서 결측인 사람들의 원본값을 대조한다.

In [8]:
derived_df = usable_df[usable_df['변수유형']=='파생(변화율)']
print(derived_df['결측률(%)'].describe())
print(f"\n결측률 50% 이상: {(derived_df['결측률(%)']>=50).sum()}개")
print(f"결측률 80% 이상: {(derived_df['결측률(%)']>=80).sum()}개")

count    161.000000
mean      36.869441
std       37.547817
min        0.000000
25%        0.000000
50%       46.110000
75%       77.410000
max       88.930000
Name: 결측률(%), dtype: float64

결측률 50% 이상: 77개
결측률 80% 이상: 30개


In [9]:
# 결측 원인 확인 - QOQ 변수 하나 골라서 원본값과 대조
check_col = 'QOQ_R3M_M_CONV_AMT_RTC'
orig_col = 'R3M_M_CONV_AMT'  # 대응하는 원본 추정

df_check = pd.read_csv(df_path, usecols=['CUST_ID', check_col, orig_col])

missing_mask = df_check[check_col].isnull()
print(f"{check_col} 결측 인원: {missing_mask.sum()}명")
print(f"\n결측인 사람들의 원본({orig_col}) 값 분포:")
print(df_check.loc[missing_mask, orig_col].describe())

print(f"\n결측 아닌 사람들의 원본 값 분포:")
print(df_check.loc[~missing_mask, orig_col].describe())

QOQ_R3M_M_CONV_AMT_RTC 결측 인원: 292693명

결측인 사람들의 원본(R3M_M_CONV_AMT) 값 분포:
count    292693.000000
mean          0.650590
std          10.459906
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         270.000000
Name: R3M_M_CONV_AMT, dtype: float64

결측 아닌 사람들의 원본 값 분포:
count    37307.000000
mean       105.568151
std        105.152451
min          0.000000
25%         19.000000
50%         39.000000
75%        240.000000
max        292.000000
Name: R3M_M_CONV_AMT, dtype: float64


In [10]:
# 방법: 결측을 '변화없음/활동없음'을 뜻하는 0으로 채우고,
# 원래 결측이었는지 여부를 별도 플래그 컬럼으로 남김

df_check[f'{check_col}_was_missing'] = df_check[check_col].isnull().astype(int)
df_check[check_col] = df_check[check_col].fillna(0)

print(df_check[[check_col, f'{check_col}_was_missing']].describe())

       QOQ_R3M_M_CONV_AMT_RTC  QOQ_R3M_M_CONV_AMT_RTC_was_missing
count           330000.000000                       330000.000000
mean                -0.007177                            0.886948
std                  0.223038                            0.316656
min                 -1.000000                            0.000000
25%                  0.000000                            1.000000
50%                  0.000000                            1.000000
75%                  0.000000                            1.000000
max                 24.444444                            1.000000


In [11]:
# 몇 개 다른 QOQ/YOY 변수로 같은 패턴 확인
sample_cols = ['QOQ_R3M_BEAUTY_AMT_RTC', 'YOY_YR_SALES_RTC', 'CPYT_TRANS_CPPY_AMT_RT']

for col in sample_cols:
    # 대응 원본변수 추정 (접두사 제거)
    orig_guess = col.replace('QOQ_', '').replace('YOY_', '').replace('CPYT_', '').replace('_RTC','').replace('_RT','').replace('_CPPY','')
    print(f"\n=== {col} (추정 원본: {orig_guess}) ===")
    try:
        df_tmp = pd.read_csv(df_path, usecols=['CUST_ID', col, orig_guess])
        missing_mask = df_tmp[col].isnull()
        print(f"결측 인원: {missing_mask.sum()}명")
        print(f"결측인 사람들 원본 중앙값: {df_tmp.loc[missing_mask, orig_guess].median()}")
        print(f"결측아닌 사람들 원본 중앙값: {df_tmp.loc[~missing_mask, orig_guess].median()}")
    except Exception as e:
        print(f"원본 컬럼명 추정 실패 또는 오류: {e}")


=== QOQ_R3M_BEAUTY_AMT_RTC (추정 원본: R3M_BEAUTY_AMT) ===
결측 인원: 293458명
결측인 사람들 원본 중앙값: 0.0
결측아닌 사람들 원본 중앙값: 49.0

=== YOY_YR_SALES_RTC (추정 원본: YR_SALES) ===
결측 인원: 291368명
결측인 사람들 원본 중앙값: 0.0
결측아닌 사람들 원본 중앙값: 95762.5

=== CPYT_TRANS_CPPY_AMT_RT (추정 원본: TRANS_AMT) ===
원본 컬럼명 추정 실패 또는 오류: Usecols do not match columns, columns expected but not found: ['TRANS_AMT']


In [12]:
sheet11[sheet11['필드명']=='CPYT_TRANS_CPPY_AMT_RT']

,No,필드명,설명,코드여부,타입
451,452,CPYT_TRANS_CPPY_AMT_RT,전년총액대비_교통_전년동기이용금액비중,N,Num


In [13]:
df_check2 = pd.read_csv(df_path, usecols=['CUST_ID', 'CPYT_TRANS_CPPY_AMT_RT', 'R3M_TRANS_AMT'])

missing_mask = df_check2['CPYT_TRANS_CPPY_AMT_RT'].isnull()
print(f"결측 인원: {missing_mask.sum()}명")
print(f"결측인 사람들 원본(R3M_TRANS_AMT) 중앙값: {df_check2.loc[missing_mask, 'R3M_TRANS_AMT'].median()}")
print(f"결측아닌 사람들 원본 중앙값: {df_check2.loc[~missing_mask, 'R3M_TRANS_AMT'].median()}")

결측 인원: 289115명
결측인 사람들 원본(R3M_TRANS_AMT) 중앙값: 0.0
결측아닌 사람들 원본 중앙값: 40.0


**결과**: 결측은 측정 실패가 아니라 "해당 소비·활동 자체가 없어 증감률 계산 대상이 없음"을 뜻하는 구조적 결측으로 확인되었다. 세 계열 모두 결측인 사람들의 원본값 중앙값이 0이었다.

## 4. 파생변수 결측값 일괄 처리

파생변수 161개는 결측을 0으로 채우고, 원래 결측 여부를 나타내는 _was_missing 플래그 161개를 별도 생성한다.

In [14]:
derived_cols = usable_df[usable_df['변수유형']=='파생(변화율)']['변수명'].tolist()

df_derived = pd.read_csv(df_path, usecols=['CUST_ID'] + derived_cols)

# 결측 플래그 먼저 생성 (원본 결측 여부 기록)
flag_cols = {}
for col in derived_cols:
    flag_cols[f'{col}_was_missing'] = df_derived[col].isnull().astype(int)

flag_df = pd.DataFrame(flag_cols)

# 그 다음 원본 파생변수는 0으로 채움
df_derived_filled = df_derived[derived_cols].fillna(0)

print(f"파생변수 데이터: {df_derived_filled.shape}")
print(f"플래그 데이터: {flag_df.shape}")
print(f"\n결측 처리 후 값 확인:")
print(df_derived_filled.isnull().sum().sum(), "개 결측 (0이어야 정상)")

파생변수 데이터: (330000, 161)
플래그 데이터: (330000, 161)

결측 처리 후 값 확인:
0 개 결측 (0이어야 정상)


## 5. 파생변수 차원축소(PCA) 시도

161개 파생변수를 표준화 후 PCA로 압축 가능한지 시도한다.

In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_derived_filled)

# 일단 넉넉하게 주성분을 뽑아서 누적 설명분산을 확인
pca = PCA(n_components=30, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
cumulative = explained.cumsum()

for i in [5, 10, 15, 20, 25, 30]:
    print(f"주성분 {i}개까지 누적 설명분산: {cumulative[i-1]*100:.1f}%")

주성분 5개까지 누적 설명분산: 23.8%
주성분 10개까지 누적 설명분산: 31.6%
주성분 15개까지 누적 설명분산: 38.0%
주성분 20개까지 누적 설명분산: 43.7%
주성분 25개까지 누적 설명분산: 48.7%
주성분 30개까지 누적 설명분산: 53.3%


**결과**: 주성분 30개로도 누적 설명분산 53.3%에 그쳐 압축 효율이 낮은 것으로 판단하고 PCA 방식을 포기하였다. 이후 개별 변수 IV 기반 선별로 전환한다.

## 6. X 후보 통합 및 학습/적용 모집단 분리

원본 197개, 파생 161개, 결측 플래그 161개를 통합하고, 씬파일러 정의로 신용이력 보유군/씬파일러를 분리한 뒤 202203의 PYE_D1011000C를 타겟(y)으로 병합한다.

In [16]:
# 원본 197개 컬럼 목록
orig_cols = usable_df[usable_df['변수유형']=='원본']['변수명'].tolist()

# 202103에서 원본 변수 + 씬파일러 정의 필드 로드
df_orig = pd.read_csv(df_path, usecols=['CUST_ID'] + orig_cols)

# 앞서 만든 파생변수(결측처리완료), 플래그와 CUST_ID 기준으로 합치기
df_derived_filled['CUST_ID'] = df_derived['CUST_ID'].values
flag_df['CUST_ID'] = df_derived['CUST_ID'].values

X_all = df_orig.merge(df_derived_filled, on='CUST_ID').merge(flag_df, on='CUST_ID')
print(f"X 후보 전체: {X_all.shape}")  # (330000, 1+197+161+161) = (330000, 520) 예상

# 씬파일러 정의 필드 다시 불러와서 세그먼트 구분
thin_def_cols = ['PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004', 'PYE_C18233005', 'PYE_L10210000']
df_def = pd.read_csv(df_path, usecols=['CUST_ID'] + thin_def_cols)

thin_mask = (
    (df_def['PYE_C1M210000']==0) & (df_def['PYE_C18233003']==0) &
    (df_def['PYE_C18233004']==0) & (df_def['PYE_C18233005']==0) &
    (df_def['PYE_L10210000']==0)
)
df_def['segment'] = thin_mask.map({True: 'thin_filer', False: 'credit_holder'})

# y 로드 (202203의 D1011000C)
df_y = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=['CUST_ID', 'PYE_D1011000C'])
df_y['y'] = (df_y['PYE_D1011000C'] > 0).astype(int)

# 전부 CUST_ID 기준으로 병합
full_data = X_all.merge(df_def[['CUST_ID','segment']], on='CUST_ID').merge(df_y[['CUST_ID','y']], on='CUST_ID')

credit_holders_final = full_data[full_data['segment']=='credit_holder'].copy()
thin_filers_final = full_data[full_data['segment']=='thin_filer'].copy()

print(f"\n신용이력 보유군: {len(credit_holders_final):,}명")
print(f"양성(y=1): {credit_holders_final['y'].sum():,}명 ({credit_holders_final['y'].mean()*100:.3f}%)")
print(f"씬파일러: {len(thin_filers_final):,}명")

X 후보 전체: (330000, 520)

신용이력 보유군: 285,890명
양성(y=1): 12,088명 (4.228%)
씬파일러: 44,110명


## 7. IV(정보가치) 계산 함수 정의 및 1차 계산

WOE/IV 계산 함수를 정의하고, 519개 후보 전체에 대해 1차 IV를 계산한다.

In [17]:
def calc_iv(df, feature, target, bins=10):
    """연속형/이산형 변수의 IV를 계산"""
    data = df[[feature, target]].copy()

    # 고유값이 적으면(범주형 성격) 그대로 그룹화, 많으면 구간화
    if data[feature].nunique() <= bins:
        data['bin'] = data[feature]
    else:
        try:
            data['bin'] = pd.qcut(data[feature], q=bins, duplicates='drop')
        except Exception:
            return None  # 구간화 실패 시 (예: 값이 거의 상수인 경우)

    grouped = data.groupby('bin')[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']

    total_bad = grouped['bad'].sum()
    total_good = grouped['good'].sum()

    # 0 방지용 스무딩
    grouped['bad_rate'] = (grouped['bad'] + 0.5) / (total_bad + 0.5)
    grouped['good_rate'] = (grouped['good'] + 0.5) / (total_good + 0.5)

    grouped['woe'] = np.log(grouped['good_rate'] / grouped['bad_rate'])
    grouped['iv'] = (grouped['good_rate'] - grouped['bad_rate']) * grouped['woe']

    return grouped['iv'].sum()

# 테스트로 하나만 먼저 계산해보기
test_col = orig_cols[0]
print(f"{test_col} IV: {calc_iv(credit_holders_final, test_col, 'y')}")

SEX IV: 0.0017034029229122754


In [18]:
exclude_from_iv = ['CUST_ID', 'segment', 'y', 'PYE_D1011000C']  # y 계산에 쓰인 원본 컬럼도 혹시 섞였으면 제외
all_x_cols = [c for c in credit_holders_final.columns if c not in exclude_from_iv]

print(f"IV 계산 대상: {len(all_x_cols)}개")

iv_results = []
for col in all_x_cols:
    try:
        iv = calc_iv(credit_holders_final, col, 'y')
        iv_results.append({'변수명': col, 'IV': iv})
    except Exception as e:
        iv_results.append({'변수명': col, 'IV': None})

iv_df = pd.DataFrame(iv_results).sort_values('IV', ascending=False)
print(f"\nIV 계산 실패(None): {iv_df['IV'].isnull().sum()}개")
print(f"\n=== IV 상위 20개 ===")
print(iv_df.head(20))

IV 계산 대상: 519개


/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 


IV 계산 실패(None): 1개

=== IV 상위 20개 ===
                                 변수명        IV
50                     PYE_C1L120012  0.854728
53                     PYE_C1L120237  0.692944
52                     PYE_C1L120196  0.665119
57                     PYE_L102100CP  0.381986
56                     PYE_L1021006P  0.375277
55                     PYE_L1021003P  0.369006
467  YOY_CRDT_LN_BAL_RTC_was_missing  0.350947
413  QOQ_CRDT_LN_BAL_RTC_was_missing  0.336555
61                     PYE_L10210800  0.332399
66                     PYE_L10216800  0.265936
252              QOQ_CRDT_LN_BAL_RTC  0.256915
306              YOY_CRDT_LN_BAL_RTC  0.247475
54                     PYE_C1L120250  0.226872
60                     PYE_L90210300  0.165112
130                              DAR  0.158123
59                     PYE_L90210200  0.154288
71                     PYE_AL012G019  0.149575
63                     PYE_L10216000  0.134829
30                          TOT_ASST  0.134258
69                   

**결과**: 상위권이 PYE_C1L120012(신용카드총한도금액) 등 신용거래 보유 수준을 직접 재는 변수들로 채워지고 IV가 0.6~0.9대로 비정상적으로 높게 나타났다. 데이터 누출(노출 변수와 타겟의 구조적 연관)이 의심되었다.


## 8. 신용 노출 관련 변수 식별 및 제외

노출 수준(카드·대출 보유)과 연체율 사이의 관계를 확인하고, 코드북에서 노출 관련 필드를 식별하여 제외한다.

In [19]:
# 신용 노출과 직접 관련된 필드군 확인 (코드북 기반)
exposure_keywords = ['L102', 'L1021', 'C1L120', 'L90210', 'L10216', 'CRDT_LN_BAL', 'HOUS_LN_BAL',
                       'SHP_LN_BAL', 'C1M2B', 'CD_USE_AMT']

exposure_related = sheet11[sheet11['필드명'].str.contains('|'.join(exposure_keywords), na=False, regex=True)]
print(f"노출 관련 후보 필드: {len(exposure_related)}개")
exposure_related[['필드명','설명']]

노출 관련 후보 필드: 40개


,필드명,설명
41,PYE_ALL_CD_USE_AMT,전년도 연간전체카드소비금액
55,HOUS_LN_BAL,주택담보대출잔액
56,CRDT_LN_BAL,신용대출잔액
57,CD_USE_AMT,카드소비금액
58,HOUS_LN_BAL_NEW,분기신규주담대출여부
59,CRDT_LN_BAL_NEW,분기신규신용대출여부
66,SHP_LN_BAL,사업자대출잔액
69,PYE_C1M2B4W03,3개월내신용카드일시불총이용금액(해지포함)_전년말기준
70,PYE_C1M2B5W03,3개월내신용카드할부총이용금액(해지포함)_전년말기준
75,PYE_C1L120012,신용카드총한도금액(활성카드)(미해지)_전년말기준


In [20]:
exposure_fields = exposure_related['필드명'].tolist()

# was_missing 플래그도 같이 제외 (원본이 노출 관련이면 플래그도 노출 정보를 담고 있음)
exposure_flag_fields = [f'{col}_was_missing' for col in exposure_fields if f'{col}_was_missing' in credit_holders_final.columns]

all_exclude = exclude_from_iv + exposure_fields + exposure_flag_fields
final_x_cols = [c for c in credit_holders_final.columns if c not in all_exclude]

print(f"제외 후 최종 IV 계산 대상: {len(final_x_cols)}개")

iv_results2 = []
for col in final_x_cols:
    try:
        iv = calc_iv(credit_holders_final, col, 'y')
        iv_results2.append({'변수명': col, 'IV': iv})
    except Exception:
        iv_results2.append({'변수명': col, 'IV': None})

iv_df2 = pd.DataFrame(iv_results2).sort_values('IV', ascending=False)
print(f"IV 계산 실패: {iv_df2['IV'].isnull().sum()}개")
print(f"\n=== IV 상위 20개 (노출 변수 제외 후) ===")
print(iv_df2.head(20))

제외 후 최종 IV 계산 대상: 480개


/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 

IV 계산 실패: 1개

=== IV 상위 20개 (노출 변수 제외 후) ===
                        변수명        IV
105                     DAR  0.158123
46            PYE_AL012G019  0.149575
29                 TOT_ASST  0.134258
44            PYE_AL012G005  0.122464
25             PYE_NET_ASST  0.119167
43            PYE_L90216300  0.118269
272        YOY_TOT_ASST_RTC  0.114760
70   R3M_ITRT_ENT_BROADCAST  0.084345
23                  PYE_ICM  0.080146
15                   PET_GD  0.072207
24               PYE_ICM_RT  0.067524
11                   APP_GD  0.065382
1                       AGE  0.061922
8                   LIF_STG  0.061602
33             OWN_HOUS_CNT  0.060205
106                     ROP  0.060162
14                  GOLF_GD  0.059377
35         FAM_OWN_HOUS_CNT  0.058813
16                TRAVEL_GD  0.058159
27           PYE_CAR_LN_AMT  0.055932


In [21]:
# 놓친 노출 관련 필드 재검색 - 더 넓은 패턴
extra_exposure = sheet11[sheet11['필드명'].str.contains('L9021|L1021|LN_', na=False, regex=True)]
print(extra_exposure[['필드명','설명']])

# IV 하위 분포 확인
print(iv_df2['IV'].describe())
print(f"\nIV < 0.02 (제외 대상): {(iv_df2['IV']<0.02).sum()}개")
print(f"IV 0.02~0.1 (약함): {((iv_df2['IV']>=0.02)&(iv_df2['IV']<0.1)).sum()}개")
print(f"IV >= 0.1 (중간 이상): {(iv_df2['IV']>=0.1).sum()}개")

                          필드명                                             설명
44             PYE_CAR_LN_AMT                                    전년도 차량할부약정액
55                HOUS_LN_BAL                                       주택담보대출잔액
56                CRDT_LN_BAL                                         신용대출잔액
58            HOUS_LN_BAL_NEW                                     분기신규주담대출여부
59            CRDT_LN_BAL_NEW                                     분기신규신용대출여부
66                 SHP_LN_BAL                                        사업자대출잔액
80              PYE_L10210000                                대출건수(미해지)_전년말기준
81              PYE_L1021003P                            3개월전대출건수(미해지)_전년말기준
82              PYE_L1021006P                            6개월전대출건수(미해지)_전년말기준
83              PYE_L102100CP                           12개월전대출건수(미해지)_전년말기준
84              PYE_L90210100                            은행업종대출건수(미해지)_전년말기준
85              PYE_L90210200                            카드업종대출건수(미해지)_전년말기준

In [22]:
# 1. 노출변수 완전히 제외한 최종 리스트로 재계산
final_exposure_exclude = [
    'PYE_CAR_LN_AMT', 'HOUS_LN_BAL', 'CRDT_LN_BAL', 'HOUS_LN_BAL_NEW', 'CRDT_LN_BAL_NEW',
    'SHP_LN_BAL', 'PYE_L10210000', 'PYE_L1021003P', 'PYE_L1021006P', 'PYE_L102100CP',
    'PYE_L90210100', 'PYE_L90210200', 'PYE_L90210300', 'PYE_L10210800', 'PYE_L10210B00',
    'PYE_L10216000', 'PYE_L90216100', 'PYE_L90216300', 'PYE_L10216800', 'PYE_L10216B00',
    'PYE_L1021A000', 'QOQ_HOUS_LN_BAL_RTC', 'QOQ_CRDT_LN_BAL_RTC', 'QOQ_HOUS_LN_BAL_NEW_CHYN',
    'QOQ_CRDT_LN_BAL_NEW_CHYN', 'QOQ_SHP_LN_BAL_RTC', 'YOY_HOUS_LN_BAL_RTC', 'YOY_CRDT_LN_BAL_RTC',
    'YOY_HOUS_LN_BAL_NEW_CHYN', 'YOY_CRDT_LN_BAL_NEW_CHYN', 'YOY_SHP_LN_BAL_RTC'
]

# 2. IV 0.1 이상인 7개, 0.02~0.1인 54개 목록을 직접 확인
print("=== IV 0.1 이상 (7개) ===")
print(iv_df2[iv_df2['IV']>=0.1][['변수명','IV']])

print("\n=== IV 0.02~0.1 (54개) 중 상위 20개 ===")
print(iv_df2[(iv_df2['IV']>=0.02)&(iv_df2['IV']<0.1)].head(20)[['변수명','IV']])

=== IV 0.1 이상 (7개) ===
                  변수명        IV
105               DAR  0.158123
46      PYE_AL012G019  0.149575
29           TOT_ASST  0.134258
44      PYE_AL012G005  0.122464
25       PYE_NET_ASST  0.119167
43      PYE_L90216300  0.118269
272  YOY_TOT_ASST_RTC  0.114760

=== IV 0.02~0.1 (54개) 중 상위 20개 ===
                        변수명        IV
70   R3M_ITRT_ENT_BROADCAST  0.084345
23                  PYE_ICM  0.080146
15                   PET_GD  0.072207
24               PYE_ICM_RT  0.067524
11                   APP_GD  0.065382
1                       AGE  0.061922
8                   LIF_STG  0.061602
33             OWN_HOUS_CNT  0.060205
106                     ROP  0.060162
14                  GOLF_GD  0.059377
35         FAM_OWN_HOUS_CNT  0.058813
16                TRAVEL_GD  0.058159
27           PYE_CAR_LN_AMT  0.055932
26              PYE_CAR_OWN  0.052456
28             PYE_CAR_SIZE  0.050831
2                     JB_TP  0.048551
22             FST_CAR_ELPS  0.046229
7

**결과**: 노출 관련 필드 40개(이후 재검색으로 일부 추가)를 제외하고 IV를 재계산하였다. 상위권이 DAR(0.158), PYE_AL012G019(0.150), TOT_ASST(0.134) 등으로 정상화되었다.

## 9. IV ≥ 0.02 기준 1차 선정 및 다중공선성 제거

노출변수를 완전히 제외한 리스트로 IV 0.02 이상 59개를 1차 선정한다. 이후 문자형/숫자형을 구분하고, 숫자형 변수 간 상관관계 0.8 이상 쌍을 확인하여 그룹별 대표만 남긴다.

In [23]:
# 최종 노출변수 제외 리스트 (놓친 2개 추가)
final_exposure_exclude = [
    'PYE_CAR_LN_AMT', 'HOUS_LN_BAL', 'CRDT_LN_BAL', 'HOUS_LN_BAL_NEW', 'CRDT_LN_BAL_NEW',
    'SHP_LN_BAL', 'PYE_L10210000', 'PYE_L1021003P', 'PYE_L1021006P', 'PYE_L102100CP',
    'PYE_L90210100', 'PYE_L90210200', 'PYE_L90210300', 'PYE_L10210800', 'PYE_L10210B00',
    'PYE_L10216000', 'PYE_L90216100', 'PYE_L90216300', 'PYE_L10216800', 'PYE_L10216B00',
    'PYE_L1021A000', 'QOQ_HOUS_LN_BAL_RTC', 'QOQ_CRDT_LN_BAL_RTC', 'QOQ_HOUS_LN_BAL_NEW_CHYN',
    'QOQ_CRDT_LN_BAL_NEW_CHYN', 'QOQ_SHP_LN_BAL_RTC', 'YOY_HOUS_LN_BAL_RTC', 'YOY_CRDT_LN_BAL_RTC',
    'YOY_HOUS_LN_BAL_NEW_CHYN', 'YOY_CRDT_LN_BAL_NEW_CHYN', 'YOY_SHP_LN_BAL_RTC'
]

# IV 0.02 이상 & 노출변수 아닌 것만 최종 채택
final_selected = iv_df2[
    (iv_df2['IV'] >= 0.02) & (~iv_df2['변수명'].isin(final_exposure_exclude))
].sort_values('IV', ascending=False)

print(f"최종 선정 변수: {len(final_selected)}개")
final_selected

최종 선정 변수: 59개


,변수명,IV
105,DAR,0.158123
46,PYE_AL012G019,0.149575
29,TOT_ASST,0.134258
44,PYE_AL012G005,0.122464
25,PYE_NET_ASST,0.119167
272,YOY_TOT_ASST_RTC,0.114760
70,R3M_ITRT_ENT_BROADCAST,0.084345
23,PYE_ICM,0.080146
15,PET_GD,0.072207
24,PYE_ICM_RT,0.067524


In [24]:
final_cols = final_selected['변수명'].tolist()
# 59개 중 실제 타입 확인
dtype_check = credit_holders_final[final_cols].dtypes
print(dtype_check)

# object(문자형)인 컬럼만 추출
object_cols = dtype_check[dtype_check == 'object'].index.tolist()
print(f"\n문자형(범주형) 변수: {len(object_cols)}개")
print(object_cols)

# 숫자형만 추출
numeric_cols = [c for c in final_cols if c not in object_cols]
print(f"\n숫자형 변수: {len(numeric_cols)}개")

DAR                                      float64
PYE_AL012G019                              int64
TOT_ASST                                   int64
PYE_AL012G005                              int64
PYE_NET_ASST                             float64
YOY_TOT_ASST_RTC                         float64
R3M_ITRT_ENT_BROADCAST                     int64
PYE_ICM                                  float64
PET_GD                                    object
PYE_ICM_RT                               float64
APP_GD                                    object
AGE                                        int64
LIF_STG                                    int64
OWN_HOUS_CNT                               int64
ROP                                        int64
GOLF_GD                                   object
FAM_OWN_HOUS_CNT                           int64
TRAVEL_GD                                 object
PYE_CAR_OWN                                int64
PYE_CAR_SIZE                               int64
JB_TP               

In [25]:
numeric_cols = [c for c in final_cols if c not in object_cols]

corr_matrix = credit_holders_final[numeric_cols].corr().abs()

high_corr_pairs = []
for i in range(len(numeric_cols)):
    for j in range(i+1, len(numeric_cols)):
        c = corr_matrix.iloc[i, j]
        if c >= 0.8:
            high_corr_pairs.append((numeric_cols[i], numeric_cols[j], round(c, 3)))

high_corr_df = pd.DataFrame(high_corr_pairs, columns=['변수1', '변수2', '상관계수']).sort_values('상관계수', ascending=False)
print(f"상관계수 0.8 이상 쌍: {len(high_corr_df)}개")
high_corr_df

상관계수 0.8 이상 쌍: 30개


,변수1,변수2,상관계수
5,CPYT_CONV_CPPY_AMT_RT_was_missing,CPYT_CONV_CPPY_CUM_AMT_RT_was_missing,1.000
3,OWN_HOUS_CNT,ROP,1.000
10,CPYT_CONV_CPPY_CUM_AMT_RT_was_missing,CPYT_CONV_CUM_AMT_RT_was_missing,1.000
9,CPYT_CONV_CPPY_CUM_AMT_RT_was_missing,CPYT_CONV_AMT_RT_was_missing,1.000
7,CPYT_CONV_CPPY_AMT_RT_was_missing,CPYT_CONV_CUM_AMT_RT_was_missing,1.000
6,CPYT_CONV_CPPY_AMT_RT_was_missing,CPYT_CONV_AMT_RT_was_missing,1.000
12,CPYT_CONV_AMT_RT_was_missing,CPYT_CONV_CUM_AMT_RT_was_missing,1.000
27,CPYT_FOOD_AMT_RT_was_missing,CPYT_FOOD_CPPY_CUM_AMT_RT_was_missing,1.000
22,CPYT_FOOD_CUM_AMT_RT_was_missing,CPYT_FOOD_CPPY_AMT_RT_was_missing,1.000
23,CPYT_FOOD_CUM_AMT_RT_was_missing,CPYT_FOOD_AMT_RT_was_missing,1.000


In [26]:
# 상관관계로 제거할 변수 목록 (그룹별 대표만 남기고 나머지 제거)
drop_by_corr = [
    'CPYT_CONV_CPPY_AMT_RT_was_missing', 'CPYT_CONV_CPPY_CUM_AMT_RT_was_missing',
    'CPYT_CONV_CUM_AMT_RT_was_missing', 'YOY_R3M_CONV_AMT_RTC_was_missing',
    'CPYT_FOOD_CUM_AMT_RT_was_missing', 'CPYT_FOOD_CPPY_AMT_RT_was_missing',
    'CPYT_FOOD_CPPY_CUM_AMT_RT_was_missing',
    'R3M_MART_AMT', 'R9M_MART_AMT', 'R12M_MART_AMT',
    'PYE_NET_ASST', 'ROP', 'PYE_CAR_SIZE', 'B1Y_EQP_MDL', 'LIF_STG',
    'YOY_R3M_ITRT_FIN_INSUR_CS', 'R3M_ITRT_COMM_SNS', 'R6M_MED_AMT',
]

# 최종 확정 (숫자형 55개 + 문자형 4개 - 상관관계 제거 18개)
final_confirmed = [c for c in final_cols if c not in drop_by_corr]
print(f"최종 확정 변수: {len(final_confirmed)}개")
print(final_confirmed)

최종 확정 변수: 41개
['DAR', 'PYE_AL012G019', 'TOT_ASST', 'PYE_AL012G005', 'YOY_TOT_ASST_RTC', 'R3M_ITRT_ENT_BROADCAST', 'PYE_ICM', 'PET_GD', 'PYE_ICM_RT', 'APP_GD', 'AGE', 'OWN_HOUS_CNT', 'GOLF_GD', 'FAM_OWN_HOUS_CNT', 'TRAVEL_GD', 'PYE_CAR_OWN', 'JB_TP', 'FST_CAR_ELPS', 'OWN_LIV_YN', 'FST_HOUS_BUY_ELPS', 'FAM_OWN_LIV_YN', 'RET_SIL', 'CPYT_CONV_AMT_RT_was_missing', 'YOY_R3M_FOOD_AMT_RTC', 'R6M_MART_AMT', 'YOY_R3M_SSM_AMT_RTC', 'HIGHEND_CD2', 'QOQ_TOT_ASST_RTC', 'PYE_AL012G011', 'SHP_CNT', 'R3M_ITRT_SHOP_JIKGU', 'YOY_R3M_MART_AMT_RTC', 'B1Y_MOB_OS', 'CPYT_FOOD_AMT_RT_was_missing', 'QOQ_R3M_MART_AMT_RTC', 'HOME_ADM', 'YOY_R3M_HOUSEHOLD_AMT_RTC', 'R3M_ITRT_ENT_WEBTOON', 'R12M_MED_AMT', 'R3M_ITRT_FIN_INSUR', 'YOY_R3M_ITRT_COMM_VOIP_CS']


**결과**: 상관계수 0.8 이상 쌍 30개를 확인하고, 각 그룹에서 IV 최고값만 남겨 41개로 정리하였다.

## 10. PYE_ 접두사 필드의 순환논리 재검증

"PYE_ 접두사는 전부 제외"라는 팀 문서의 원칙을 일괄 적용하지 않고, 씬파일러 정의 필드와의 상관계수를 실측하여 개별 판단한다.

In [27]:
thin_def_cols = ['PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004', 'PYE_C18233005', 'PYE_L10210000']
suspect_cols = ['PYE_AL012G005', 'PYE_AL012G019', 'PYE_AL012G011', 'PYE_ICM', 'PYE_ICM_RT', 'PYE_CAR_OWN']

check_cols = ['CUST_ID'] + thin_def_cols + suspect_cols
df_check = pd.read_csv(df_path, usecols=check_cols)

# 씬파일러 정의 필드들을 하나로 합친 '총 노출건수' 만들어서 비교 (더 직관적)
df_check['total_exposure'] = (
    df_check['PYE_C1M210000'] + df_check['PYE_C18233003'] +
    df_check['PYE_C18233004'] + df_check['PYE_C18233005'] + df_check['PYE_L10210000']
)

print("=== 씬파일러 정의 필드(총노출) vs 의심 변수들의 상관계수 ===")
for col in suspect_cols:
    corr = df_check['total_exposure'].corr(df_check[col])
    print(f"{col}: {corr:.4f}")

=== 씬파일러 정의 필드(총노출) vs 의심 변수들의 상관계수 ===
PYE_AL012G005: 0.2665
PYE_AL012G019: 0.1318
PYE_AL012G011: 0.2850
PYE_ICM: 0.4170
PYE_ICM_RT: -0.4657
PYE_CAR_OWN: 0.2063


In [28]:
final_confirmed_v2 = [c for c in final_confirmed if c not in ['PYE_ICM', 'PYE_ICM_RT']]
print(f"업데이트된 최종 변수: {len(final_confirmed_v2)}개")

업데이트된 최종 변수: 39개


**결과**: PYE_AL012G005/011/019(이력변경), PYE_CAR_OWN(차량보유)은 상관계수 0.13~0.29로 낮아 유지하였다. PYE_ICM(소득, 0.417), PYE_ICM_RT(소득백분위, -0.466)는 상관관계가 높아 제외하였다. 39개로 정리되었다.

## 11. R3M_ITRT_*(관심도) 35개 재검증

팀 문서가 35개 전부 채택을 권장하였으나 실제로는 4개만 IV 기준을 통과한 상태였다. 고유값·최빈값 비율을 확인하여 원인이 계산 오류가 아닌지 검증한다.

In [29]:
r3m_itrt_cols = [c for c in sheet11['필드명'] if c.startswith('R3M_ITRT_')]
print(f"R3M_ITRT_* 전체: {len(r3m_itrt_cols)}개")

df_itrt = pd.read_csv(df_path, usecols=['CUST_ID'] + r3m_itrt_cols)

# 각 변수의 고유값 개수와 실제 값 몇 개 확인
itrt_check = []
for col in r3m_itrt_cols:
    n_unique = df_itrt[col].nunique()
    sample_values = sorted(df_itrt[col].dropna().unique())[:10]
    itrt_check.append({'변수명': col, '고유값수': n_unique, '값예시': sample_values})

itrt_check_df = pd.DataFrame(itrt_check)
print(itrt_check_df)

R3M_ITRT_* 전체: 35개
                        변수명  고유값수           값예시
0          R3M_ITRT_FIN_PAY     2        [0, 1]
1        R3M_ITRT_FIN_INSUR     2        [0, 1]
2         R3M_ITRT_FIN_COIN     2        [0, 1]
3         R3M_ITRT_FIN_BANK     3     [0, 1, 2]
4        R3M_ITRT_FIN_ASSET     2        [0, 1]
5        R3M_ITRT_FIN_STOCK     2        [0, 1]
6      R3M_ITRT_LIFE_HEALTH     2        [0, 1]
7    R3M_ITRT_LIFE_DELIVERY     3     [0, 1, 2]
8       R3M_ITRT_LIFE_HOUSE     2        [0, 1]
9        R3M_ITRT_LIFE_MOVE     2        [0, 1]
10        R3M_ITRT_LIFE_CAR     2        [0, 1]
11       R3M_ITRT_LIFE_KIDS     2        [0, 1]
12     R3M_ITRT_LIFE_BEAUTY     2        [0, 1]
13       R3M_ITRT_SHOP_MART     2        [0, 1]
14     R3M_ITRT_SHOP_SOCIAL     3     [0, 1, 2]
15       R3M_ITRT_SHOP_OPEN     3     [0, 1, 2]
16     R3M_ITRT_SHOP_BEAUTY     3     [0, 1, 2]
17      R3M_ITRT_SHOP_JIKGU     2        [0, 1]
18   R3M_ITRT_LEISURE_SPORT     2        [0, 1]
19  R3M_ITRT_LEISURE_

In [30]:
# 35개 전체 실제 IV 값과 최빈값 비율을 나란히 확인
r3m_iv_check = []
for col in r3m_itrt_cols:
    try:
        iv = calc_iv(credit_holders_final if col in credit_holders_final.columns else df_itrt.merge(credit_holders_final[['CUST_ID','y']], on='CUST_ID'), col, 'y')
    except Exception as e:
        iv = None
    top_ratio = df_itrt[col].value_counts(normalize=True).iloc[0] * 100
    r3m_iv_check.append({'변수명': col, 'IV': iv, '최빈값비율(%)': round(top_ratio,2)})

r3m_iv_df = pd.DataFrame(r3m_iv_check).sort_values('IV', ascending=False)
print(r3m_iv_df.to_string())

                        변수명        IV  최빈값비율(%)
22   R3M_ITRT_ENT_BROADCAST  0.084345     43.14
31        R3M_ITRT_COMM_SNS  0.044114     40.40
17      R3M_ITRT_SHOP_JIKGU  0.023825     60.25
25     R3M_ITRT_ENT_WEBTOON  0.021081     83.86
1        R3M_ITRT_FIN_INSUR  0.020163     56.09
14     R3M_ITRT_SHOP_SOCIAL  0.018862     50.64
21        R3M_ITRT_ENT_SVOD  0.018565     58.08
34  R3M_ITRT_COMM_MESSENGER  0.017544     44.47
7    R3M_ITRT_LIFE_DELIVERY  0.015936     50.71
30     R3M_ITRT_INFO_PORTAL  0.014940     58.65
16     R3M_ITRT_SHOP_BEAUTY  0.014838     50.94
28       R3M_ITRT_INFO_BOOK  0.012561     61.21
3         R3M_ITRT_FIN_BANK  0.010581     59.21
11       R3M_ITRT_LIFE_KIDS  0.009154     57.45
13       R3M_ITRT_SHOP_MART  0.008965     74.07
2         R3M_ITRT_FIN_COIN  0.008073     83.10
4        R3M_ITRT_FIN_ASSET  0.006613     97.52
26       R3M_ITRT_ENT_MUSIC  0.006291     58.58
33       R3M_ITRT_COMM_DATE  0.006234     95.49
9        R3M_ITRT_LIFE_MOVE  0.006219   

**결과**: 쏠림 정도와 IV 사이에 뚜렷한 상관관계가 없어, 나머지 31개는 실제로 연체와의 관련성이 낮은 것으로 판단하고 기존 필터링 결과를 유지하였다.

## 12. 팀 문서에만 있던 누락 변수 재확인

HIGHEND_CD1, HIGHEND_CD3, CAR_YN, VIP_CARD_YN, COM_ADM, HOME_ADM 6개가 최종 목록에서 빠진 이유를 확인한다.

In [31]:
check_missing_vars = ['HIGHEND_CD1', 'HIGHEND_CD3', 'CAR_YN', 'VIP_CARD_YN', 'COM_ADM', 'HOME_ADM']

for v in check_missing_vars:
    in_all = v in all_fields
    in_candidate = v in candidate_fields
    in_usable = v in usable
    print(f"{v}: 전체필드존재={in_all}, 후보포함={in_candidate}, 1차사용가능={in_usable}")

HIGHEND_CD1: 전체필드존재=True, 후보포함=True, 1차사용가능=True
HIGHEND_CD3: 전체필드존재=True, 후보포함=True, 1차사용가능=True
CAR_YN: 전체필드존재=True, 후보포함=True, 1차사용가능=True
VIP_CARD_YN: 전체필드존재=True, 후보포함=True, 1차사용가능=True
COM_ADM: 전체필드존재=True, 후보포함=True, 1차사용가능=True
HOME_ADM: 전체필드존재=True, 후보포함=True, 1차사용가능=True


In [32]:
check_missing_vars = ['HIGHEND_CD1', 'HIGHEND_CD3', 'CAR_YN', 'VIP_CARD_YN', 'COM_ADM', 'HOME_ADM']

for v in check_missing_vars:
    try:
        iv = calc_iv(credit_holders_final, v, 'y') if v in credit_holders_final.columns else None
        if iv is None:
            print(f"{v}: credit_holders_final에 컬럼 없음 - 별도 로드 필요")
        else:
            print(f"{v}: IV = {iv:.4f}")
    except Exception as e:
        print(f"{v}: 계산 오류 - {e}")

HIGHEND_CD1: IV = 0.0049
HIGHEND_CD3: IV = 0.0119
CAR_YN: IV = 0.0103
VIP_CARD_YN: IV = 0.0058
COM_ADM: IV = 0.0192
HOME_ADM: IV = 0.0220


/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])


In [33]:
# COM_ADM 원본 대신 '직장정보 존재여부' 이진 플래그로 변환해서 IV 재계산
df_com = pd.read_csv(df_path, usecols=['CUST_ID', 'COM_ADM'])
df_com = df_com.merge(credit_holders_final[['CUST_ID','y']], on='CUST_ID')

df_com['has_workplace'] = df_com['COM_ADM'].notna().astype(int)

missing_rate = df_com['COM_ADM'].isnull().mean() * 100
print(f"COM_ADM 결측률(=직장정보 없음 비율): {missing_rate:.2f}%")

iv_flag = calc_iv(df_com, 'has_workplace', 'y')
print(f"has_workplace(이진 플래그) IV: {iv_flag:.4f}")

COM_ADM 결측률(=직장정보 없음 비율): 12.37%
has_workplace(이진 플래그) IV: 0.0066


In [34]:
final_confirmed_v3 = final_confirmed_v2 + ['HOME_ADM']
# 중복 체크
final_confirmed_v3 = list(dict.fromkeys(final_confirmed_v3))
print(f"업데이트된 최종 변수: {len(final_confirmed_v3)}개")

업데이트된 최종 변수: 39개


**결과**: HOME_ADM(IV 0.022)만 컷라인을 넘겨 추가하였다(40개). COM_ADM은 원본과 '직장정보 존재여부' 이진 플래그 버전 모두 IV가 낮아 제외를 유지하였다.

## 13. 팀 제안 신규 파생변수의 원본 재료 검증

팀 문서가 제안한 파생변수(필수·재량 소비 구성비, 생활 리듬 안정성, 자기계발 지수 등)의 재료가 되는 원본 컬럼들의 IV를 개별 확인한다.

In [35]:
building_blocks = [
    # B-3 DISC_RATIO 재료
    'R12M_ENT_AMT', 'R12M_HOTEL_AMT', 'R12M_DEP_AMT', 'R12M_FOOD_AMT',
    'R12M_TRANS_AMT', 'R12M_MART_AMT', 'R12M_CONV_AMT', 'BUY_LUX_YN',
    # B-5 생활리듬
    'R3M_OUTSIDE_WDAY_CNT', 'R3M_HOME_WDAY_STAY_TIME', 'R3M_MOV_WDAY_DIST',
    'QOQ_R3M_MOV_WDAY_DIST_RTC', 'QOQ_R3M_HOME_WDAY_STAY_TIME_RTC',
    # B-7 주거자산 (ROP, OWN_LIV_YN은 이미 확인됨)
    'FAM_OWN_LIV_YN',
    # B-8 금융스트레스
    'R3M_ITRT_FIN_PAY', 'R3M_ITRT_FIN_COIN',
    'QOQ_R3M_ITRT_FIN_PAY_CS', 'QOQ_R3M_ITRT_FIN_COIN_CS',
    # B-10 디지털결제
    'MUSIC_GD', 'GAME_GD',
    # B-12 자기계발
    'R12M_EDU_AMT', 'YOY_R3M_EDU_AMT_RTC', 'YOY_R3M_ITRT_INFO_BOOK_CS',
    # B-13 일상소비
    'R3M_SSM_AMT', 'R3M_CONV_AMT',
]

df_blocks = pd.read_csv(df_path, usecols=['CUST_ID'] + building_blocks)
df_blocks = df_blocks.merge(credit_holders_final[['CUST_ID','y']], on='CUST_ID')

block_iv = []
for col in building_blocks:
    try:
        iv = calc_iv(df_blocks, col, 'y')
    except Exception as e:
        iv = None
    block_iv.append({'변수명': col, 'IV': iv})

block_iv_df = pd.DataFrame(block_iv).sort_values('IV', ascending=False, na_position='last')
print(block_iv_df.to_string())

/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 

                                변수명        IV
21              YOY_R3M_EDU_AMT_RTC  0.122791
13                   FAM_OWN_LIV_YN  0.042072
5                     R12M_MART_AMT  0.023540
6                     R12M_CONV_AMT  0.018849
19                          GAME_GD  0.016719
22        YOY_R3M_ITRT_INFO_BOOK_CS  0.014194
24                     R3M_CONV_AMT  0.011112
3                     R12M_FOOD_AMT  0.010123
15                R3M_ITRT_FIN_COIN  0.008073
16          QOQ_R3M_ITRT_FIN_PAY_CS  0.007831
17         QOQ_R3M_ITRT_FIN_COIN_CS  0.007764
23                      R3M_SSM_AMT  0.007626
14                 R3M_ITRT_FIN_PAY  0.004278
18                         MUSIC_GD  0.000059
0                      R12M_ENT_AMT  0.000000
1                    R12M_HOTEL_AMT  0.000000
4                    R12M_TRANS_AMT  0.000000
8              R3M_OUTSIDE_WDAY_CNT  0.000000
7                        BUY_LUX_YN  0.000000
2                      R12M_DEP_AMT  0.000000
12  QOQ_R3M_HOME_WDAY_STAY_TIME_RT

/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 

In [36]:
# IV=0으로 나온 변수들 값 분포 직접 확인
zero_iv_cols = ['R12M_ENT_AMT', 'R12M_HOTEL_AMT', 'R12M_DEP_AMT', 'R12M_TRANS_AMT',
                 'R12M_EDU_AMT', 'R3M_OUTSIDE_WDAY_CNT', 'R3M_HOME_WDAY_STAY_TIME',
                 'R3M_MOV_WDAY_DIST', 'BUY_LUX_YN', 'QOQ_R3M_HOME_WDAY_STAY_TIME_RTC',
                 'QOQ_R3M_MOV_WDAY_DIST_RTC']

for col in zero_iv_cols:
    print(f"\n{col}:")
    print(df_blocks[col].describe())
    print(f"고유값 수: {df_blocks[col].nunique()}")


R12M_ENT_AMT:
count    285890.000000
mean          0.018699
std           1.947678
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         387.000000
Name: R12M_ENT_AMT, dtype: float64
고유값 수: 31

R12M_HOTEL_AMT:
count    285890.000000
mean          0.035626
std           6.364392
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        1748.000000
Name: R12M_HOTEL_AMT, dtype: float64
고유값 수: 14

R12M_DEP_AMT:
count    285890.000000
mean         34.953300
std         133.812858
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        1986.000000
Name: R12M_DEP_AMT, dtype: float64
고유값 수: 1176

R12M_TRANS_AMT:
count    285890.000000
mean         14.137165
std          46.922629
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         380.000000
Name: R12M_TRANS_AMT, dtype: float64
고유값 수: 288

R12M_EDU_AMT

**중간 결과**: R3M_OUTSIDE_WDAY_CNT 등 5개는 결측 100%로 데이터 자체가 없어 '생활 리듬 안정성' 파생변수를 폐기하였다. R12M_ENT_AMT 등 소비 재료 변수들은 IV가 정확히 0으로 나와 계산 방식 자체를 점검할 필요가 확인되었다.

## 14. IV 계산 로직 개선(치우친 분포 대응) 및 재검토

값의 대부분이 0에 몰린 변수에서 균등 구간화(qcut)가 실패해 IV가 0으로 왜곡되는 문제를 발견하였다. 0 여부를 먼저 분리하는 방식으로 함수를 개선하고, 기존 IV가 매우 낮았던 변수 전체를 재검토한다.

In [37]:
def calc_iv_v2(df, feature, target, bins=10):
    data = df[[feature, target]].copy()

    if data[feature].nunique() <= bins:
        data['bin'] = data[feature].astype(str)
    else:
        # 0과 0 초과를 먼저 분리
        is_zero = data[feature] == 0
        data['bin'] = 'zero'
        nonzero = data.loc[~is_zero, feature]
        if nonzero.nunique() > 1:
            try:
                nonzero_bins = pd.qcut(nonzero, q=min(bins-1, nonzero.nunique()), duplicates='drop')
                data.loc[~is_zero, 'bin'] = nonzero_bins.astype(str)
            except Exception:
                data.loc[~is_zero, 'bin'] = 'nonzero'

    grouped = data.groupby('bin', observed=True)[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']

    total_bad = grouped['bad'].sum()
    total_good = grouped['good'].sum()

    grouped['bad_rate'] = (grouped['bad'] + 0.5) / (total_bad + 0.5)
    grouped['good_rate'] = (grouped['good'] + 0.5) / (total_good + 0.5)

    grouped['woe'] = np.log(grouped['good_rate'] / grouped['bad_rate'])
    grouped['iv'] = (grouped['good_rate'] - grouped['bad_rate']) * grouped['woe']

    return grouped['iv'].sum()

# 재계산
for col in ['R12M_ENT_AMT', 'R12M_HOTEL_AMT', 'R12M_DEP_AMT', 'R12M_TRANS_AMT', 'R12M_EDU_AMT']:
    iv_old = calc_iv(df_blocks, col, 'y')
    iv_new = calc_iv_v2(df_blocks, col, 'y')
    print(f"{col}: 기존방식={iv_old:.4f} → 개선방식={iv_new:.4f}")

R12M_ENT_AMT: 기존방식=0.0000 → 개선방식=0.0016


/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 

R12M_HOTEL_AMT: 기존방식=0.0000 → 개선방식=0.0006
R12M_DEP_AMT: 기존방식=0.0000 → 개선방식=0.0086
R12M_TRANS_AMT: 기존방식=0.0000 → 개선방식=0.0160
R12M_EDU_AMT: 기존방식=0.0000 → 개선방식=0.0011


/tmp/ipykernel_5140/2715139179.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])


In [38]:
# 기존 IV 결과(iv_df2)에서 매우 낮은 값(0.001 이하)만 재검토 대상으로 추출
low_iv_candidates = iv_df2[iv_df2['IV'] <= 0.001]['변수명'].tolist()
print(f"재검토 대상(기존 IV≤0.001): {len(low_iv_candidates)}개")

recheck_results = []
for col in low_iv_candidates:
    try:
        iv_new = calc_iv_v2(credit_holders_final, col, 'y')
        recheck_results.append({'변수명': col, 'IV_개선': iv_new})
    except Exception:
        recheck_results.append({'변수명': col, 'IV_개선': None})

recheck_df = pd.DataFrame(recheck_results).sort_values('IV_개선', ascending=False)
print(f"\n개선 후 IV 0.02 이상으로 올라온 변수: {(recheck_df['IV_개선']>=0.02).sum()}개")
print(recheck_df.head(20))

재검토 대상(기존 IV≤0.001): 160개

개선 후 IV 0.02 이상으로 올라온 변수: 6개
                        변수명     IV_개선
4      YOY_R3M_CONV_AMT_RTC  0.034632
56           R6M_M_MART_AMT  0.022281
60           R9M_M_MART_AMT  0.021622
79           R3M_M_MART_AMT  0.021314
92          R12M_M_MART_AMT  0.021115
27   YOY_R3M_M_CONV_AMT_RTC  0.020698
61         R6M_M_MART_E_AMT  0.016734
87           R12M_TRANS_AMT  0.016040
76         R3M_M_MART_E_AMT  0.015974
37           R3M_M_CONV_AMT  0.013965
80         R9M_M_MART_E_AMT  0.013962
75        R12M_M_MART_E_AMT  0.013707
73               SHP_CD_USE  0.012537
51            R9M_TRANS_AMT  0.011681
152           R6M_TRANS_AMT  0.011254
72         R3M_M_MART_H_AMT  0.011132
84         R9M_M_MART_H_AMT  0.010794
7            R9M_E_COMM_AMT  0.010749
49         R6M_M_MART_H_AMT  0.010520
83        R12M_M_MART_H_AMT  0.010422


In [39]:
new_candidates = ['YOY_R3M_CONV_AMT_RTC', 'R6M_M_MART_AMT', 'R9M_M_MART_AMT',
                   'R3M_M_MART_AMT', 'R12M_M_MART_AMT', 'YOY_R3M_M_CONV_AMT_RTC']

df_new_check = pd.read_csv(df_path, usecols=['CUST_ID'] + new_candidates)
corr_new = df_new_check[new_candidates].corr().abs()
print(corr_new.round(3))

                        YOY_R3M_CONV_AMT_RTC  R6M_M_MART_AMT  R9M_M_MART_AMT  \
YOY_R3M_CONV_AMT_RTC                   1.000           0.090           0.088   
R6M_M_MART_AMT                         0.090           1.000           0.992   
R9M_M_MART_AMT                         0.088           0.992           1.000   
R3M_M_MART_AMT                         0.091           0.982           0.972   
R12M_M_MART_AMT                        0.085           0.983           0.994   
YOY_R3M_M_CONV_AMT_RTC                 0.406           0.091           0.089   

                        R3M_M_MART_AMT  R12M_M_MART_AMT  \
YOY_R3M_CONV_AMT_RTC             0.091            0.085   
R6M_M_MART_AMT                   0.982            0.983   
R9M_M_MART_AMT                   0.972            0.994   
R3M_M_MART_AMT                   1.000            0.962   
R12M_M_MART_AMT                  0.962            1.000   
YOY_R3M_M_CONV_AMT_RTC           0.089            0.086   

                        Y

In [40]:
final_additions = ['YOY_R3M_CONV_AMT_RTC', 'R6M_M_MART_AMT', 'YOY_R3M_M_CONV_AMT_RTC']

final_confirmed_v4 = list(dict.fromkeys(final_confirmed_v3 + final_additions))
print(f"최종 확정 변수: {len(final_confirmed_v4)}개")

최종 확정 변수: 42개


**결과**: 160개 재검토 중 6개가 새로 0.02 이상으로 확인되었다. M_MART_AMT 계열 4개는 상관계수 0.96 이상으로 중복 확인되어 대표 1개(R6M_M_MART_AMT)만 채택하고, CONV 계열 2개(상관계수 0.406)는 모두 유지하여 42개로 정리되었다.

## 15. IV 계산 실패 변수 원인 확인

계산 실패했던 유일한 변수(COFFEE_1ST)의 원인을 확인한다.

In [41]:
failed_var = iv_df2[iv_df2['IV'].isnull()]['변수명'].tolist()
print(f"계산 실패 변수: {failed_var}")

# 왜 실패했는지 직접 확인
for col in failed_var:
    print(f"\n{col}:")
    print(credit_holders_final[col].describe())
    print(f"고유값 수: {credit_holders_final[col].nunique()}")
    print(f"결측 수: {credit_holders_final[col].isnull().sum()}")

계산 실패 변수: ['COFFEE_1ST']

COFFEE_1ST:
count     285890
unique        16
top            *
freq      182925
Name: COFFEE_1ST, dtype: object
고유값 수: 16
결측 수: 0


In [42]:
# COFFEE_1ST 값 분포 전체 확인
print(credit_holders_final['COFFEE_1ST'].value_counts())

# '*' 제외하고 IV 계산 (범주형이니 그대로 그룹화)
def calc_iv_categorical(df, feature, target):
    data = df[[feature, target]].copy()
    grouped = data.groupby(feature, observed=True)[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']
    total_bad = grouped['bad'].sum()
    total_good = grouped['good'].sum()
    grouped['bad_rate'] = (grouped['bad'] + 0.5) / (total_bad + 0.5)
    grouped['good_rate'] = (grouped['good'] + 0.5) / (total_good + 0.5)
    grouped['woe'] = np.log(grouped['good_rate'] / grouped['bad_rate'])
    grouped['iv'] = (grouped['good_rate'] - grouped['bad_rate']) * grouped['woe']
    return grouped['iv'].sum()

iv_coffee = calc_iv_categorical(credit_holders_final, 'COFFEE_1ST', 'y')
print(f"\nCOFFEE_1ST IV (범주형 처리): {iv_coffee:.4f}")

COFFEE_1ST
*     182925
01     27657
12     21295
21     13721
02     11672
16     11160
05     10038
22      4367
23      1282
20       900
04       712
13       112
19        21
24        19
06         8
18         1
Name: count, dtype: int64

COFFEE_1ST IV (범주형 처리): 0.0023


In [43]:
# 혹시 생긴 중복 컬럼(_x, _y) 정리
dup_check = [c for c in credit_holders_final.columns if c.endswith('_x') or c.endswith('_y')]
print(f"중복으로 생긴 컬럼: {dup_check}")

# 원래 컬럼이 이미 있었는지 직접 확인
for col in ['HOME_ADM', 'YOY_R3M_CONV_AMT_RTC', 'R6M_M_MART_AMT', 'YOY_R3M_M_CONV_AMT_RTC']:
    exists = col in credit_holders_final.columns
    print(f"{col}: 이미 존재={exists}")

중복으로 생긴 컬럼: []
HOME_ADM: 이미 존재=True
YOY_R3M_CONV_AMT_RTC: 이미 존재=True
R6M_M_MART_AMT: 이미 존재=True
YOY_R3M_M_CONV_AMT_RTC: 이미 존재=True


In [44]:
for col in ['HOME_ADM', 'HOME_ADM_x', 'HOME_ADM_y',
            'R6M_M_MART_AMT', 'R6M_M_MART_AMT_x', 'R6M_M_MART_AMT_y',
            'YOY_R3M_CONV_AMT_RTC', 'YOY_R3M_CONV_AMT_RTC_x', 'YOY_R3M_CONV_AMT_RTC_y',
            'YOY_R3M_M_CONV_AMT_RTC', 'YOY_R3M_M_CONV_AMT_RTC_x', 'YOY_R3M_M_CONV_AMT_RTC_y']:
    print(f"{col}: {col in credit_holders_final.columns}")

HOME_ADM: True
HOME_ADM_x: False
HOME_ADM_y: False
R6M_M_MART_AMT: True
R6M_M_MART_AMT_x: False
R6M_M_MART_AMT_y: False
YOY_R3M_CONV_AMT_RTC: True
YOY_R3M_CONV_AMT_RTC_x: False
YOY_R3M_CONV_AMT_RTC_y: False
YOY_R3M_M_CONV_AMT_RTC: True
YOY_R3M_M_CONV_AMT_RTC_x: False
YOY_R3M_M_CONV_AMT_RTC_y: False


**결과**: 문자형 변수로 qcut이 적용되지 않아 실패했던 것으로 확인되었다. 범주형 방식으로 재계산한 결과 IV 0.0023으로 매우 낮아 채택하지 않았다.

## 16. 42개 확정 목록 재현

런타임 재시작 이후에도 동일한 상태로 이어갈 수 있도록, 지금까지 확정된 42개 변수 목록을 재확인한다.

In [45]:
final_confirmed_v4 = ['DAR', 'PYE_AL012G019', 'TOT_ASST', 'PYE_AL012G005', 'YOY_TOT_ASST_RTC',
                       'R3M_ITRT_ENT_BROADCAST', 'PET_GD', 'APP_GD', 'AGE', 'OWN_HOUS_CNT',
                       'GOLF_GD', 'FAM_OWN_HOUS_CNT', 'TRAVEL_GD', 'PYE_CAR_OWN', 'JB_TP',
                       'FST_CAR_ELPS', 'OWN_LIV_YN', 'FST_HOUS_BUY_ELPS', 'FAM_OWN_LIV_YN',
                       'RET_SIL', 'CPYT_CONV_AMT_RT_was_missing', 'YOY_R3M_FOOD_AMT_RTC',
                       'R6M_MART_AMT', 'YOY_R3M_SSM_AMT_RTC', 'HIGHEND_CD2', 'QOQ_TOT_ASST_RTC',
                       'PYE_AL012G011', 'SHP_CNT', 'R3M_ITRT_SHOP_JIKGU', 'YOY_R3M_MART_AMT_RTC',
                       'B1Y_MOB_OS', 'CPYT_FOOD_AMT_RT_was_missing', 'QOQ_R3M_MART_AMT_RTC',
                       'HOME_ADM', 'YOY_R3M_HOUSEHOLD_AMT_RTC', 'R3M_ITRT_ENT_WEBTOON',
                       'R12M_MED_AMT', 'R3M_ITRT_FIN_INSUR', 'YOY_R3M_ITRT_COMM_VOIP_CS',
                       'YOY_R3M_CONV_AMT_RTC', 'R6M_M_MART_AMT', 'YOY_R3M_M_CONV_AMT_RTC']

print(f"최종 변수 개수: {len(final_confirmed_v4)}개")

# 전부 존재하는지 확인
missing = [c for c in final_confirmed_v4 if c not in credit_holders_final.columns]
print(f"누락된 변수: {missing}")

최종 변수 개수: 42개
누락된 변수: []


In [46]:
print(f"최종 확정 변수: {len(final_confirmed_v4)}개")
print(final_confirmed_v4)

# 타입 재확인
dtype_final = credit_holders_final[final_confirmed_v4].dtypes if all(c in credit_holders_final.columns for c in final_confirmed_v4) else None

최종 확정 변수: 42개
['DAR', 'PYE_AL012G019', 'TOT_ASST', 'PYE_AL012G005', 'YOY_TOT_ASST_RTC', 'R3M_ITRT_ENT_BROADCAST', 'PET_GD', 'APP_GD', 'AGE', 'OWN_HOUS_CNT', 'GOLF_GD', 'FAM_OWN_HOUS_CNT', 'TRAVEL_GD', 'PYE_CAR_OWN', 'JB_TP', 'FST_CAR_ELPS', 'OWN_LIV_YN', 'FST_HOUS_BUY_ELPS', 'FAM_OWN_LIV_YN', 'RET_SIL', 'CPYT_CONV_AMT_RT_was_missing', 'YOY_R3M_FOOD_AMT_RTC', 'R6M_MART_AMT', 'YOY_R3M_SSM_AMT_RTC', 'HIGHEND_CD2', 'QOQ_TOT_ASST_RTC', 'PYE_AL012G011', 'SHP_CNT', 'R3M_ITRT_SHOP_JIKGU', 'YOY_R3M_MART_AMT_RTC', 'B1Y_MOB_OS', 'CPYT_FOOD_AMT_RT_was_missing', 'QOQ_R3M_MART_AMT_RTC', 'HOME_ADM', 'YOY_R3M_HOUSEHOLD_AMT_RTC', 'R3M_ITRT_ENT_WEBTOON', 'R12M_MED_AMT', 'R3M_ITRT_FIN_INSUR', 'YOY_R3M_ITRT_COMM_VOIP_CS', 'YOY_R3M_CONV_AMT_RTC', 'R6M_M_MART_AMT', 'YOY_R3M_M_CONV_AMT_RTC']


## 17. 문자형 변수 인코딩

최종 후보 중 PET_GD, APP_GD, GOLF_GD, TRAVEL_GD 4개는 문자형이다. 값 분포와 특수값('*')의 성격을 확인한다.

In [47]:
for col in ['PET_GD', 'APP_GD', 'GOLF_GD', 'TRAVEL_GD']:
    print(f"\n{col} 값 분포:")
    print(credit_holders_final[col].value_counts())


PET_GD 값 분포:
PET_GD
0    223236
*     42716
3      8307
2      6925
1      4706
Name: count, dtype: int64

APP_GD 값 분포:
APP_GD
0    224352
*     42716
3      8516
2      5927
1      4379
Name: count, dtype: int64

GOLF_GD 값 분포:
GOLF_GD
0    230462
*     42716
3      7075
2      4004
1      1633
Name: count, dtype: int64

TRAVEL_GD 값 분포:
TRAVEL_GD
0    224318
*     42716
3      8526
2      5807
1      4523
Name: count, dtype: int64


In [48]:
mask_pet = credit_holders_final['PET_GD'] == '*'
mask_app = credit_holders_final['APP_GD'] == '*'
mask_golf = credit_holders_final['GOLF_GD'] == '*'
mask_travel = credit_holders_final['TRAVEL_GD'] == '*'

print(f"PET='*' & APP='*' 겹치는 인원: {(mask_pet & mask_app).sum()}")
print(f"전부 동시에 '*'인 인원: {(mask_pet & mask_app & mask_golf & mask_travel).sum()}")

PET='*' & APP='*' 겹치는 인원: 42716
전부 동시에 '*'인 인원: 42716


In [49]:
credit_holders_final['grade_unavailable'] = (credit_holders_final['PET_GD'] == '*').astype(int)

iv_flag = calc_iv(credit_holders_final, 'grade_unavailable', 'y')
print(f"grade_unavailable(등급산정불가 플래그) IV: {iv_flag:.4f}")

# 참고로 이 사람들이 씬파일러와도 겹치는지 확인 (개념적 이해를 위해)
thin_filers_final['grade_unavailable'] = (thin_filers_final['PET_GD'] == '*').astype(int)
print(f"\n신용이력보유군 내 등급산정불가 비율: {credit_holders_final['grade_unavailable'].mean()*100:.2f}%")
print(f"씬파일러 내 등급산정불가 비율: {thin_filers_final['grade_unavailable'].mean()*100:.2f}%")

grade_unavailable(등급산정불가 플래그) IV: 0.0579

신용이력보유군 내 등급산정불가 비율: 14.94%
씬파일러 내 등급산정불가 비율: 33.65%


**중간 결과**: 4개 변수의 '*'가 정확히 동일한 42,716명을 가리켜, 개별 결측이 아니라 공통된 세그먼트(등급 산정 불가 집단)를 의미하는 것으로 확인되었다. 이진 플래그(grade_unavailable)의 IV는 0.0579였다.

In [50]:
def woe_encode(df, feature, target):
    data = df[[feature, target]].copy()
    grouped = data.groupby(feature, observed=True)[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']
    total_bad = grouped['bad'].sum()
    total_good = grouped['good'].sum()
    grouped['bad_rate'] = (grouped['bad'] + 0.5) / (total_bad + 0.5)
    grouped['good_rate'] = (grouped['good'] + 0.5) / (total_good + 0.5)
    grouped['woe'] = np.log(grouped['good_rate'] / grouped['bad_rate'])
    return grouped['woe'].to_dict()

for col in ['PET_GD', 'APP_GD', 'GOLF_GD', 'TRAVEL_GD']:
    woe_map = woe_encode(credit_holders_final, col, 'y')
    credit_holders_final[f'{col}_woe'] = credit_holders_final[col].map(woe_map)
    thin_filers_final[f'{col}_woe'] = thin_filers_final[col].map(woe_map)
    print(f"{col} WOE 매핑: {woe_map}")

PET_GD WOE 매핑: {'*': -0.4989032963588266, '0': 0.16060445883920585, '1': -0.32280349812564274, '2': -0.305286859508266, '3': -0.24473601718343588}
APP_GD WOE 매핑: {'*': -0.4989032963588266, '0': 0.1424363034596288, '1': 0.026837515678247454, '2': -0.011778546404543077, '3': -0.3185652080508549}
GOLF_GD WOE 매핑: {'*': -0.4989032963588266, '0': 0.11723824421533847, '1': 0.3884034438681558, '2': -0.13965187715600683, '3': 0.2014922938457029}
TRAVEL_GD WOE 매핑: {'*': -0.4989032963588266, '0': 0.11452721844511002, '1': 0.07215504854332203, '2': 0.09354293558903891, '3': 0.21041355504437906}


In [51]:
def woe_encode_nonstar(df, feature, target):
    # '*' 제외하고 0~3만 대상으로 WOE 계산
    data = df[df[feature] != '*'][[feature, target]].copy()
    grouped = data.groupby(feature, observed=True)[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']
    total_bad = grouped['bad'].sum()
    total_good = grouped['good'].sum()
    grouped['bad_rate'] = (grouped['bad'] + 0.5) / (total_bad + 0.5)
    grouped['good_rate'] = (grouped['good'] + 0.5) / (total_good + 0.5)
    grouped['woe'] = np.log(grouped['good_rate'] / grouped['bad_rate'])
    return grouped['woe'].to_dict()

for col in ['PET_GD', 'APP_GD', 'GOLF_GD', 'TRAVEL_GD']:
    woe_map = woe_encode_nonstar(credit_holders_final, col, 'y')
    # '*'는 0으로(중립) 처리하고, grade_unavailable 플래그가 그 정보를 대신 담당
    credit_holders_final[f'{col}_woe'] = credit_holders_final[col].map(woe_map).fillna(0)
    thin_filers_final[f'{col}_woe'] = thin_filers_final[col].map(woe_map).fillna(0)
    print(f"{col}: {woe_map}")

# 최종 검증 - grade_unavailable + 4개 woe로 IV 재확인
for col in ['PET_GD_woe', 'APP_GD_woe', 'GOLF_GD_woe', 'TRAVEL_GD_woe']:
    iv = calc_iv(credit_holders_final, col, 'y')
    print(f"{col} (재분리 후) IV: {iv:.4f}")

PET_GD: {'0': 0.04401157516221488, '1': -0.43939638180263324, '2': -0.4218797431852567, '3': -0.36132890086042646}
APP_GD: {'0': 0.02584341978263822, '1': -0.08975536799874348, '2': -0.12837143008153384, '3': -0.43515809172784564}
GOLF_GD: {'0': 0.0006453605383477592, '1': 0.27181056019116523, '2': -0.2562447608329975, '3': 0.0848994101687122}
TRAVEL_GD: {'0': -0.0020656652318807186, '1': -0.04443783513366839, '2': -0.02304994808795173, '3': 0.09382067136738835}
PET_GD_woe (재분리 후) IV: 0.0722
APP_GD_woe (재분리 후) IV: 0.0654
GOLF_GD_woe (재분리 후) IV: 0.0594
TRAVEL_GD_woe (재분리 후) IV: 0.0582


**결과**: '*' 여부를 grade_unavailable 플래그로 분리하고, 0~3 등급만을 대상으로 WOE를 재계산하였다. 재계산한 IV는 원래 값과 거의 동일하여(PET 0.072→0.072 등), 원래 IV의 대부분이 등급 자체의 차이에서 비롯된 것임을 확인하였다.

In [52]:
# 원본 문자형 4개 제거하고, woe 버전 4개 + 플래그 1개 추가
final_confirmed_v5 = [c for c in final_confirmed_v4 if c not in ['PET_GD', 'APP_GD', 'GOLF_GD', 'TRAVEL_GD']]
final_confirmed_v5 += ['PET_GD_woe', 'APP_GD_woe', 'GOLF_GD_woe', 'TRAVEL_GD_woe', 'grade_unavailable']

print(f"최종 확정 변수: {len(final_confirmed_v5)}개")
print(final_confirmed_v5)

# 전부 숫자형인지 최종 확인
final_dtypes = credit_holders_final[final_confirmed_v5].dtypes
print(f"\n문자형 남아있는지: {(final_dtypes=='object').sum()}개")

최종 확정 변수: 43개
['DAR', 'PYE_AL012G019', 'TOT_ASST', 'PYE_AL012G005', 'YOY_TOT_ASST_RTC', 'R3M_ITRT_ENT_BROADCAST', 'AGE', 'OWN_HOUS_CNT', 'FAM_OWN_HOUS_CNT', 'PYE_CAR_OWN', 'JB_TP', 'FST_CAR_ELPS', 'OWN_LIV_YN', 'FST_HOUS_BUY_ELPS', 'FAM_OWN_LIV_YN', 'RET_SIL', 'CPYT_CONV_AMT_RT_was_missing', 'YOY_R3M_FOOD_AMT_RTC', 'R6M_MART_AMT', 'YOY_R3M_SSM_AMT_RTC', 'HIGHEND_CD2', 'QOQ_TOT_ASST_RTC', 'PYE_AL012G011', 'SHP_CNT', 'R3M_ITRT_SHOP_JIKGU', 'YOY_R3M_MART_AMT_RTC', 'B1Y_MOB_OS', 'CPYT_FOOD_AMT_RT_was_missing', 'QOQ_R3M_MART_AMT_RTC', 'HOME_ADM', 'YOY_R3M_HOUSEHOLD_AMT_RTC', 'R3M_ITRT_ENT_WEBTOON', 'R12M_MED_AMT', 'R3M_ITRT_FIN_INSUR', 'YOY_R3M_ITRT_COMM_VOIP_CS', 'YOY_R3M_CONV_AMT_RTC', 'R6M_M_MART_AMT', 'YOY_R3M_M_CONV_AMT_RTC', 'PET_GD_woe', 'APP_GD_woe', 'GOLF_GD_woe', 'TRAVEL_GD_woe', 'grade_unavailable']

문자형 남아있는지: 0개


**결과**: 원본 문자형 4개를 제거하고 WOE 인코딩 4개 + 플래그 1개를 추가하여 43개로 확정되었다.

## 18. 씬파일러 세그먼트 유효성 사전 점검

확정된 43개 변수가 씬파일러(44,110명)에서도 유효한 값을 갖는지, 신용이력 보유군과 최빈값 비율을 대조하여 확인한다.

In [53]:
thin_check_results = []

for col in final_confirmed_v5:
    credit_dist = credit_holders_final[col].describe()
    thin_dist = thin_filers_final[col].describe()

    # 최빈값 비율 비교
    credit_top_ratio = credit_holders_final[col].value_counts(normalize=True).iloc[0] * 100
    thin_top_ratio = thin_filers_final[col].value_counts(normalize=True).iloc[0] * 100

    thin_check_results.append({
        '변수명': col,
        '신용이력군_최빈값비율(%)': round(credit_top_ratio, 1),
        '씬파일러_최빈값비율(%)': round(thin_top_ratio, 1),
        '차이': round(thin_top_ratio - credit_top_ratio, 1)
    })

thin_check_df = pd.DataFrame(thin_check_results).sort_values('씬파일러_최빈값비율(%)', ascending=False)
print(thin_check_df.to_string())

                             변수명  신용이력군_최빈값비율(%)  씬파일러_최빈값비율(%)    차이
20                   HIGHEND_CD2            95.2           99.9   4.7
15                       RET_SIL            94.3           99.2   4.9
0                            DAR            52.0           98.8  46.8
9                    PYE_CAR_OWN            83.5           98.0  14.5
36                R6M_M_MART_AMT            93.1           97.9   4.7
23                       SHP_CNT            82.3           96.8  14.5
7                   OWN_HOUS_CNT            62.4           94.3  31.9
28          QOQ_R3M_MART_AMT_RTC            81.3           92.8  11.4
18                  R6M_MART_AMT            79.5           91.3  11.9
22                 PYE_AL012G011            53.0           90.7  37.7
25          YOY_R3M_MART_AMT_RTC            76.7           89.4  12.8
12                    OWN_LIV_YN            72.0           87.6  15.6
37        YOY_R3M_M_CONV_AMT_RTC            88.0           86.7  -1.3
30     YOY_R3M_HOUSE

In [54]:
severe_skew = thin_check_df[thin_check_df['씬파일러_최빈값비율(%)'] >= 90]['변수명'].tolist()
print(f"씬파일러 내 90% 이상 쏠림 변수 ({len(severe_skew)}개):")
print(severe_skew)

씬파일러 내 90% 이상 쏠림 변수 (10개):
['HIGHEND_CD2', 'RET_SIL', 'DAR', 'PYE_CAR_OWN', 'R6M_M_MART_AMT', 'SHP_CNT', 'OWN_HOUS_CNT', 'QOQ_R3M_MART_AMT_RTC', 'R6M_MART_AMT', 'PYE_AL012G011']


In [55]:
# 최종 정리 - 43개 변수 + 씬파일러 해석 주의 플래그
severe_skew_vars = ['HIGHEND_CD2', 'RET_SIL', 'DAR', 'PYE_CAR_OWN', 'R6M_M_MART_AMT',
                     'SHP_CNT', 'OWN_HOUS_CNT', 'QOQ_R3M_MART_AMT_RTC', 'R6M_MART_AMT',
                     'PYE_AL012G011']

print(f"최종 X 변수: {len(final_confirmed_v5)}개")
print(f"이 중 씬파일러 적용 시 해석 주의(90%+ 쏠림): {len(severe_skew_vars)}개")
print(f"신용이력군 학습에는 문제없이 전부 사용")

# 학습/적용 데이터셋 최종 준비 상태 확인
print(f"\n신용이력 보유군(학습용): {credit_holders_final.shape}")
print(f"씬파일러(적용 전용): {thin_filers_final.shape}")
print(f"양성(y=1) 비율: {credit_holders_final['y'].mean()*100:.3f}%")

최종 X 변수: 43개
이 중 씬파일러 적용 시 해석 주의(90%+ 쏠림): 10개
신용이력군 학습에는 문제없이 전부 사용

신용이력 보유군(학습용): (285890, 527)
씬파일러(적용 전용): (44110, 527)
양성(y=1) 비율: 4.228%


**결과**: 10개 변수(DAR, HIGHEND_CD2, RET_SIL, PYE_CAR_OWN, R6M_M_MART_AMT, SHP_CNT, OWN_HOUS_CNT, QOQ_R3M_MART_AMT_RTC, R6M_MART_AMT, PYE_AL012G011)가 씬파일러 내 최빈값 비율 90% 이상으로 확인되었다. 신용이력 보유군 학습에는 영향이 없으나, 씬파일러 적용 시 해석 주의가 필요한 변수로 별도 표기하고 43개는 그대로 유지하기로 하였다.

## 19. 모델 동작 검증(스모크 테스트)

확정된 변수로 실제 로지스틱회귀가 정상 작동하는지 확인한다. 먼저 결측 여부를 점검한다.

In [56]:
missing_check = credit_holders_final[final_confirmed_v5].isnull().sum()
missing_check = missing_check[missing_check > 0]
print(missing_check)

DAR                    8421
FST_CAR_ELPS         194214
FST_HOUS_BUY_ELPS    163881
dtype: int64


In [57]:
# DAR 결측인 사람들이 자산 0인지 확인
df_dar_check = credit_holders_final[['DAR', 'TOT_ASST']].copy()
print("DAR 결측인 사람들의 TOT_ASST 분포:")
print(df_dar_check[df_dar_check['DAR'].isnull()]['TOT_ASST'].describe())

# FST_CAR_ELPS, FST_HOUS_BUY_ELPS도 같은 패턴인지 (0 근처인지는 애초에 확인필요없고, 결측 자체가 신호일 가능성)
print(f"\nFST_CAR_ELPS 결측 비율: {credit_holders_final['FST_CAR_ELPS'].isnull().mean()*100:.1f}%")
print(f"FST_HOUS_BUY_ELPS 결측 비율: {credit_holders_final['FST_HOUS_BUY_ELPS'].isnull().mean()*100:.1f}%")

DAR 결측인 사람들의 TOT_ASST 분포:
count    8421.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: TOT_ASST, dtype: float64

FST_CAR_ELPS 결측 비율: 67.9%
FST_HOUS_BUY_ELPS 결측 비율: 57.3%


**결과**: DAR, FST_CAR_ELPS, FST_HOUS_BUY_ELPS 3개에 결측이 남아있었다. DAR 결측자는 전원 총자산이 0인 사람들로, 부채비율 계산 자체가 불가능한 구조적 결측임을 확인하였다.

In [58]:
missing_cols = ['DAR', 'FST_CAR_ELPS', 'FST_HOUS_BUY_ELPS']

for col in missing_cols:
    # 결측 플래그 생성
    credit_holders_final[f'{col}_was_missing'] = credit_holders_final[col].isnull().astype(int)
    thin_filers_final[f'{col}_was_missing'] = thin_filers_final[col].isnull().astype(int)

    # 결측을 0으로 채움
    credit_holders_final[col] = credit_holders_final[col].fillna(0)
    thin_filers_final[col] = thin_filers_final[col].fillna(0)

# 결측 재확인
print(credit_holders_final[missing_cols].isnull().sum())

# 최종 X 리스트에 플래그 3개 추가
final_confirmed_v6 = final_confirmed_v5 + [f'{col}_was_missing' for col in missing_cols]
print(f"\n최종 변수 개수: {len(final_confirmed_v6)}개")

# 전체 결측 최종 확인
print(f"전체 결측 개수: {credit_holders_final[final_confirmed_v6].isnull().sum().sum()}")

DAR                  0
FST_CAR_ELPS         0
FST_HOUS_BUY_ELPS    0
dtype: int64

최종 변수 개수: 46개
전체 결측 개수: 0


**결과**: 세 변수 모두 결측을 0으로 채우고 _was_missing 플래그 3개를 추가하여 최종 46개로 확정되었다.

In [59]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

X = credit_holders_final[final_confirmed_v6]
y = credit_holders_final['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

pred_proba = model.predict_proba(X_test)[:, 1]
print(f"AUC: {roc_auc_score(y_test, pred_proba):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, pred_proba):.4f}")

coef_df = pd.DataFrame({'변수': final_confirmed_v6, '계수': model.coef_[0]}).sort_values('계수')
print(coef_df.to_string())

AUC: 0.6345
PR-AUC: 0.0664
                               변수            계수
6                             AGE -9.180311e-03
27   CPYT_FOOD_AMT_RT_was_missing -2.726713e-03
16   CPYT_CONV_AMT_RT_was_missing -2.225709e-03
8                FAM_OWN_HOUS_CNT -1.578695e-03
13              FST_HOUS_BUY_ELPS -1.532657e-03
37         YOY_R3M_M_CONV_AMT_RTC -1.450039e-03
4                YOY_TOT_ASST_RTC -1.337092e-03
35           YOY_R3M_CONV_AMT_RTC -1.288131e-03
30      YOY_R3M_HOUSEHOLD_AMT_RTC -1.218991e-03
25           YOY_R3M_MART_AMT_RTC -8.797840e-04
44       FST_CAR_ELPS_was_missing -8.767459e-04
17           YOY_R3M_FOOD_AMT_RTC -8.531386e-04
36                 R6M_M_MART_AMT -8.479739e-04
18                   R6M_MART_AMT -6.155608e-04
19            YOY_R3M_SSM_AMT_RTC -5.289320e-04
38                     PET_GD_woe -5.089783e-04
20                    HIGHEND_CD2 -4.648732e-04
15                        RET_SIL -4.515038e-04
7                    OWN_HOUS_CNT -3.795995e-04
21           

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


**중간 결과**: 최적화 알고리즘이 수렴하지 못했다는 경고가 발생하였다. 스케일 차이로 인한 문제로 판단하여 표준화를 적용한다.

In [60]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model2 = LogisticRegression(max_iter=2000, class_weight='balanced')
model2.fit(X_train_scaled, y_train)

pred_proba2 = model2.predict_proba(X_test_scaled)[:, 1]
print(f"AUC: {roc_auc_score(y_test, pred_proba2):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, pred_proba2):.4f}")

AUC: 0.7073
PR-AUC: 0.1112


In [61]:
# 경고 없이 수렴했는지 확인 - 별다른 warning 없이 이 셀이 조용히 끝나면 정상
print("수렴 완료")
print(f"최종 AUC: {roc_auc_score(y_test, pred_proba2):.4f}")
print(f"최종 PR-AUC: {average_precision_score(y_test, pred_proba2):.4f}")

# 계수 방향성 재확인 (스케일링 후 버전)
coef_df2 = pd.DataFrame({'변수': final_confirmed_v6, '계수': model2.coef_[0]}).sort_values('계수', key=abs, ascending=False)
print(coef_df2.head(15))

수렴 완료
최종 AUC: 0.7073
최종 PR-AUC: 0.1112
                              변수        계수
27  CPYT_FOOD_AMT_RT_was_missing -0.280868
1                  PYE_AL012G019  0.252238
44      FST_CAR_ELPS_was_missing -0.241099
42             grade_unavailable  0.238202
4               YOY_TOT_ASST_RTC -0.183186
18                  R6M_MART_AMT -0.167872
32                  R12M_MED_AMT -0.162615
2                       TOT_ASST -0.148838
20                   HIGHEND_CD2 -0.141486
5         R3M_ITRT_ENT_BROADCAST  0.132824
3                  PYE_AL012G005  0.128039
26                    B1Y_MOB_OS  0.114000
16  CPYT_CONV_AMT_RT_was_missing -0.104999
0                            DAR  0.102720
11                  FST_CAR_ELPS -0.098127


**최종 결과**: 표준화 적용 후 AUC 0.7073, PR-AUC 0.1112로 확인되었다. 이는 프로젝트 기획서의 목표 수준(정보가 풍부한 환경에서 AUC 70% 이상)에 부합한다. 계수 방향성도 상식과 일치하였다(주소·연락처 변경 이력이 잦을수록 위험 증가, 자산이 많을수록 위험 감소 등).